# Hub API — Guide & Cookbook

Runnable reference for every endpoint exposed by the unified-hub app.

**Setup (once):** make sure your SSH tunnel is up — `ssh -fN vision` (assumes `~/.ssh/config` has `LocalForward 8800 127.0.0.1:8800` on `Vision` host via the `dc2` ProxyJump).

Then just edit the params in the next cell and run top-to-bottom.

In [ ]:
# ── Params ──────────────────────────────────────────────────────────────────
HUB         = "http://localhost:8800"   # tunnel endpoint 
FILE        = "/path/to/report.html"    # any HTML you want to upload
COLLECTION  = "Funding Reports"         # collection name (created if missing)
COLOR       = "#06b6d4"                 # collection accent
TAGS        = ["funding", "auto"]       # arbitrary tag list
DESCRIPTION = "daily funding snapshot"  # free-form notes (optional)
LINK_URL    = "https://lindenshore.atlassian.net/wiki/spaces/AZD" # example - can add any url you want, or leave blank
LINK_NAME   = "Darius Confluence Homepage" # free-form name for the above url (optional)
# ────────────────────────────────────────────────────────────────────────────

import json, os, requests
from pathlib import Path

S = requests.Session()                  # reused for all calls
S.headers.update({"Accept": "application/json"})

def api(method, path, **kw):
    r = S.request(method, f"{HUB}{path}", timeout=kw.pop("timeout", 60), **kw)
    r.raise_for_status()
    return r.json() if r.content and r.headers.get("content-type","").startswith("application/json") else r

api("GET", "/api/health")               # → {'ok': True}

## 1. Collections

Collections are folders. Each has a `name`, optional `color`, optional `description`, a `position` (for ordering), and an `itemCount`.

| Method | Path | Body | Purpose |
| --- | --- | --- | --- |
| `GET`    | `/api/collections`        | — | list all + item counts |
| `POST`   | `/api/collections`        | `{name, color?, description?}` | create |
| `PATCH`  | `/api/collections/:id`    | partial fields | rename / recolor / reorder |
| `DELETE` | `/api/collections/:id`    | — | delete (items become "Unfiled") |

In [ ]:
# List collections
cols = api("GET", "/api/collections")["collections"]
for c in cols:
    print(f"  [{c['id']:>3}] {c['name']:<30} ({c['itemCount']} items)  {c['color']}")

In [ ]:
# Get (or create) a collection by name — the common pattern when uploading
def get_or_create_collection(name, color=COLOR, description=None):
    cols = api("GET", "/api/collections")["collections"]
    for c in cols:
        if c["name"] == name:
            return c
    body = {"name": name, "color": color}
    if description:
        body["description"] = description
    return api("POST", "/api/collections", json=body)

col = get_or_create_collection(COLLECTION)
col

In [ ]:
# Rename / recolor (any subset of fields)
# api("PATCH", f"/api/collections/{col['id']}", json={"name": "Funding (renamed)", "color": "#f59e0b"})

# Delete a collection (items move to Unfiled, they are NOT deleted)
# api("DELETE", f"/api/collections/{col['id']}")

## 2. Uploading files (streaming multipart — recommended)

`POST /api/items/upload` streams the request body straight to disk. Works for files of any size up to `HUB_MAX_UPLOAD_MB` (server default 5 GB). The Node process never buffers more than ~1 MB regardless of file size.

Multipart fields:

| field | required | type | notes |
| --- | --- | --- | --- |
| `file`         | ✓ | binary | the actual file bytes |
| `name`         | ✓ | string | display name in the UI |
| `collectionId` |   | int    | omit → goes to Unfiled |
| `tags`         |   | string | **JSON-encoded** array, e.g. `'["a","b"]'` |
| `description`  |   | string | free-form |
| `mimeType`     |   | string | sniffed if omitted |

In [ ]:
def upload_file(path, name=None, collection_id=None, tags=None, description=None, mime="text/html"):
    p = Path(path)
    name = name or p.name
    data = {"name": name}
    if collection_id is not None: data["collectionId"] = str(collection_id)
    if tags:                      data["tags"]         = json.dumps(tags)
    if description:               data["description"]  = description
    with open(p, "rb") as f:
        r = S.post(
            f"{HUB}/api/items/upload",
            files={"file": (name, f, mime)},
            data=data,
            timeout=600,           # bump for very large files
        )
    r.raise_for_status()
    return r.json()

item = upload_file(FILE, collection_id=col["id"], tags=TAGS, description=DESCRIPTION)
item

In [ ]:
# Bulk upload a directory of HTMLs
# from pathlib import Path
# for f in Path("/data/reports").glob("*.html"):
#     upload_file(f, collection_id=col["id"], tags=["batch"])

## 3. Adding link items

Links use the JSON endpoint `POST /api/items` with `kind="link"` — no multipart needed.

| field | required | notes |
| --- | --- | --- |
| `kind`         | ✓ | `"link"` |
| `name`         | ✓ | display name |
| `url`          | ✓ | the destination URL |
| `collectionId` |   | omit → Unfiled |
| `tags`         |   | array of strings |
| `description`  |   | free-form |

In [ ]:
link = api("POST", "/api/items", json={
    "kind": "link",
    "name": LINK_NAME,
    "url":  LINK_URL,
    "collectionId": col["id"],
    "tags": ["dashboard"],
    "description": "live grafana panel",
})
link

## 4. Listing and reading items

| Method | Path | Purpose |
| --- | --- | --- |
| `GET` | `/api/items?collection_id=N\|unfiled&kind=file\|link` | filtered metadata list |
| `GET` | `/api/items/:id`        | single item metadata |
| `GET` | `/api/items/:id/raw`    | raw HTML / file body with correct mime |
| `GET` | `/api/items/:id/raw?download=1` | force a `Content-Disposition: attachment` |
| `GET` | `/view/:id`             | HTML body for files, 302→URL for links |

In [ ]:
# All items in our collection
items = api("GET", f"/api/items?collection_id={col['id']}")["items"]
for it in items:
    print(f"  [{it['id']:>3}] {it['kind']:<4} {it['name']:<40} {it.get('size','')}")

In [ ]:
# Just files, just links
files = api("GET", f"/api/items?collection_id={col['id']}&kind=file")["items"]
links = api("GET", f"/api/items?collection_id={col['id']}&kind=link")["items"]
print(f"{len(files)} files, {len(links)} links")

# Items not assigned to any collection
unfiled = api("GET", "/api/items?collection_id=unfiled")["items"]
print(f"{len(unfiled)} unfiled")

In [ ]:
# Download the raw HTML body of an uploaded file
r = S.get(f"{HUB}/api/items/{item['id']}/raw", timeout=600, stream=True)
r.raise_for_status()
with open("/tmp/hub_download.html", "wb") as f:
    for chunk in r.iter_content(chunk_size=1 << 20):
        f.write(chunk)
print("saved:", os.path.getsize("/tmp/hub_download.html"), "bytes")

## 5. Editing items

`PATCH /api/items/:id` accepts any subset of the editable fields.

In [ ]:
# Rename, retag, move to another collection (or to Unfiled with collectionId=null)
api("PATCH", f"/api/items/{item['id']}", json={
    "name":         "renamed.html",
    "tags":         ["funding", "daily", "reviewed"],
    "description":  "updated note",
    # "collectionId": None,   # → Unfiled
    # "collectionId": other_col_id,
})

In [ ]:
# Reorder items inside a collection (drag-drop equivalent)
# Provide the full ordered list of item IDs in the collection
# api("POST", "/api/items/reorder", json={
#     "collectionId": col["id"],
#     "itemIds":      [item['id'], link['id']],
# })

## 6. Deleting

`DELETE /api/items/:id` removes the DB row AND unlinks the on-disk file.

In [ ]:
# api("DELETE", f"/api/items/{item['id']}")
# api("DELETE", f"/api/items/{link['id']}")

## 7. Browser-facing URLs

Not API calls, but useful to construct links you can share with teammates (they go through the same tunnel).

| URL | What it shows |
| --- | --- |
| `/view/:id`                                    | file body inline, or 302 redirect for link items |
| `/api/items/:id/raw?download=1`                | force-download the file |
| `/dashboard/:collectionId?cols=1\|2\|3\|4`    | server-rendered tiled iframe wall of a whole collection |
| `/#/c/:collectionId`                           | open the collection in the UI |

In [ ]:
print("Item preview :", f"{HUB}/view/{item['id']}")
print("Download URL :", f"{HUB}/api/items/{item['id']}/raw?download=1")
print("Dashboard    :", f"{HUB}/dashboard/{col['id']}?cols=2")
print("UI link      :", f"{HUB}/#/c/{col['id']}")

## 8. Error handling cheatsheet

- `413 Payload Too Large` → file exceeded `HUB_MAX_UPLOAD_MB` (bump it in pm2 env and `pm2 restart hub --update-env`)
- `400` with `{error: "..."}` → malformed body (e.g. `tags` not JSON-encoded, missing `name` on upload)
- `404` → wrong id, or tunnel hit the wrong host
- `ConnectionRefusedError` from `requests` → SSH tunnel isn't up; run `ssh -fN vision`